# ***Predictive Healthcare: Machine Learning for Disease Prediction(Diabetes)***

In [1]:
import pandas as pd        # For loading, manipulating, and exploring datasets
import numpy as np         # For numerical computations
import seaborn as sns      # For plotting graphs like count plots and heatmaps
import matplotlib.pyplot as plt      # For generating visualizations
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder   # standard scaler for feature scaling and LabelEncoder() for encoding categorical variables
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import shap       # to know how each feature contributes to the model's predictions to making the model more transparent and interpretable
from imblearn.over_sampling import SMOTE    # to balance the data (oversampling)
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as imbPipeline
from collections import Counter

# **Peoples who are not suffering from diabetes are very large as compared to others who are suffering. This shows real data**

In [ ]:
# Identify target column dynamically
# if any column name matches any name in possible_names, that column is our target. we use this to find target variable without manually mentioning it
def find_target_column(df, possible_names):
    for col in df.columns:                   # Iterate through all column names in the DataFrame
        if col.lower() in possible_names:    # Convert column name to lowercase and check if it's in the given set
            return col                       # If a match is found, return that column name
    raise ValueError(f"No suitable target column found in dataset: {df.columns}")

# Assign target column names dynamically based on dataset structure
target_col_diabetes = find_target_column(diabetes_df, {"outcome", "label", "diabetes"})

# **Label Encoding**: My dataset contains some categorial data but Machine learning models, especially those based on numerical computations, cannot process categorical so we use encoder to convert categorial data into numerical data


In [ ]:
# Encode categorical variables
# to convert categorical features into numerical form for model processing
def encode_categorical(df):
    label_encoders = {}
    for col in df.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le
    return df, label_encoders

diabetes_df, _ = encode_categorical(diabetes_df)

# **Exploratory Data Analysis (EDA)**
In this section, I will perform a detailed exploratory data analysis (EDA) to understand the
dataset's structure, features, and quality. This includes:
1. Dataset size and dimensions.
2. Types of features.
3. Statistics of features.
4. Data health assessment.
5. Handling missing data.


In [ ]:
# Remove duplicate records to avoid data processing repeatedly
diabetes_df = diabetes_df.drop_duplicates()

# **Dataset Size and Dimensions**

In [ ]:
# Dataset Information--> rows, columns in each dataset with feature types
def dataset_info(df, name):
    print(f"Dataset: {name}")
    print("Number of Rows ", df.shape[0])
    print("Number of Columns ", df.shape[1])
    print("Feature types")
    print(df.dtypes,"\n")
dataset_info(diabetes_df, "Diabetes Prediction Dataset")

# **Rows** contains Patients while **Columns** contains features
 **Types of Features**

I this section we examine the types of features in the dataset.


**It provides summary statistics for numerical features**

**including: Mean → Average value of each feature.**

**Standard Deviation → Spread of the data.**


In [ ]:
# Statistics of Features
# give summary statistics of numerical and categorical features like mean,standard deviation etc.
def feature_statistics(df, name):
    print(f"Statistics for {name}\n")
    print("Numerical Features:")
    print(df.describe(), "\n")

    categorical_cols = df.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        print("Categorical Features:")
        print(df[categorical_cols].describe(), "\n")
    else:
        print("No categorical features found after encoding.\n")

feature_statistics(diabetes_df, "Diabetes Prediction Dataset")

# **Good things about the data**
The dataset does not contain missing vlues.

Includes medically relevant attributes like BMI, Blood Glucose, HbA1c Level, and Smoking History, which are crucial for predicting diabetes.

The dataset has a reasonable number of samples, allowing the model to generalize well.


In [ ]:
# Remove Unneccessary value [0.00195%] to simplify data
diabetes_df = diabetes_df[diabetes_df['gender'] != 'Other']

In [ ]:
for column in diabetes_df.columns:                     # to know distinct values in each column
    num_distinct_values = len(diabetes_df[column].unique())
    print(f"{column}: {num_distinct_values} distinct values")

In [ ]:
# Visualizing Age Distribution
plt.hist(diabetes_df['age'], bins=30, edgecolor='black')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

# **# Helps understand the age demographics in the dataset, Identifies whether the dataset is biased toward a specific age group.**

In [ ]:
# Visualizing Gender Distribution
sns.countplot(x='gender', data=diabetes_df)
plt.title('Gender Distribution')
plt.show()

# **# Shows the male-to-female ratio in the dataset.Helps in assessing whether the dataset is balanced across genders.**

In [ ]:
# Count plots for binary variables for diabetes
for col in ['hypertension', 'heart_disease', 'diabetes']:
  sns.countplot(x=col, data=diabetes_df)
  plt.title(f'{col} Distribution')
  plt.show()

# **# Visualizes the proportion of patients with and without these conditions.Identifies class imbalance in disease presence, which is crucial for model training.**

In [ ]:
# Count plot for smoking history for diabetes
sns.countplot(x='smoking_history', data=diabetes_df)
plt.title('Smoking History Distribution')
plt.show()

# **Helps determine if smoking increases diabetes risk. Visualizes the proportion of smokers vs. non-smokers in diabetic and non-diabetic groups.**

## **# Helps determine if gender is a significant factor in diabetes prevalence.**
# **# If diabetes cases are significantly higher in one gender, gender can be an important feature in the prediction model.**

**# For diabetes = 0 (no diabetes):**

**# The median blood glucose is around 135.**

**# For diabetes = 1 (diabetes present):**

**# The median is higher, around 160.**

**This shows that people with diabetes tend to have higher blood glucose levels than those without.**

# Detects Feature Relationships (Correlation Analysis)
# It helps identify relationships between numerical features.
# Detect redundant or highly correlated features.

# **Understanding feature types helps in deciding the type of preprocessing (scaling, encoding, etc.).**

In [ ]:
# Exploratory Data Analysis (EDA)
def eda(df, name, target_col):
    # analysis the missing values
    print("Missing values:")
    print(df.isnull().sum(), "\n")

    # analysis the duplicate values
    print("Duplicate Records:", df.duplicated().sum(), "\n")

    # Plot distribution of target variable
    plt.figure(figsize=(6, 4))
    sns.countplot(x=df[target_col])
    plt.title(f"Distribution of Target Variable in {name}")
    plt.show()
    imbalance_ratio = df[target_col].value_counts(normalize=True).max()
    print(f"Class Imbalance Ratio: {imbalance_ratio:.2f} (max class proportion)")

    # Correlation Matrix
    # to know how strongly 2 variables are strongly related
    plt.figure(figsize=(12, 6))
    sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
    plt.title(f"Feature Correlation Matrix for {name}")
    plt.show()

# **This data does not contain any missing values which shows that data is healthy for use**

# **If missing values are present in a future dataset, I will replace them with the median of the respective feature to maintain data consistency.**

## **Glucose level and HbA1c level are main features to predict diabetes**

# **Data Preparation: Feature Engineering,Encoding, and Scaling**


## You use SMOTE when your dataset has a class imbalance.
What’s Class Imbalance?
**Imagine a binary classification problem where:**
**Class 0 has 91.5 % of the data**
**Class 1 has only 8.5 %**
**A model trained on this will likely just predict Class 0 all the time and still get high accuracy — but it's useless for detecting the minority class.**

## **Reduce the number of samples in the majority class by removing samples randomly**

In [ ]:
# Define resampling
over = SMOTE(sampling_strategy=0.1)
under = RandomUnderSampler(sampling_strategy=0.5)

Data preprocessing lays the groundwork for effective model training. In my workflow, I:
Standardize numerical features by subtracting their mean and dividing by their standard deviation—this centers them at zero with unit variance.
One‑hot encode categorical features so that algorithms can interpret them correctly.

**Standard scaler is a sciketlearn technique that centralize the data and make its variance 1**

In [ ]:
# Define preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level','hypertension','heart_disease']),
        ('cat', OneHotEncoder(), ['gender','smoking_history'])
    ])

# Split data into features and target variable
X = diabetes_df.drop('diabetes', axis=1)
y = diabetes_df['diabetes']

->The pipeline organizes preprocessing, resampling, and model training into one streamlined workflow, making it cleaner and more manageable.

->Oversampling (like SMOTE) and undersampling are used to balance the class distribution, which helps the model perform better on imbalanced datasets.

->A RandomForestClassifier is used as the final model for its robustness and ability to handle complex classification tasks effectively.

In [ ]:
# Create a pipeline that preprocesses the data, resamples data, and then trains a classifier
clf = imbPipeline(steps=[('preprocessor', preprocessor),
                      ('over', over),
                      ('under', under),
                      ('classifier', RandomForestClassifier())])

-> A pipeline is created to organize the workflow — it first preprocesses the data and then trains a machine learning model.

-> The model used is a RandomForestClassifier, a robust and widely-used algorithm that works well for many classification problems by building multiple decision trees and combining their results.

-> To improve the model’s performance, we use GridSearchCV from sklearn.model_selection, which:

-> Performs an exhaustive search through a grid of different combinations of hyperparameters (like number of trees, depth, etc.).

In [ ]:
# Define the hyperparameters and the values we want to test
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

In [ ]:
# Create Grid Search object
grid_search = GridSearchCV(clf, param_grid, cv=5)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
grid_search.fit(X_train, y_train)

# Print the best parameters
print("Best Parameters: ", grid_search.best_params_)

Converts **GridSearchCV** results into a DataFrame to simplify the analysis and make it easier to visualize hyperparameter tuning outcomes.

In [ ]:
# Convert GridSearchCV results to a DataFrame and plot
results_df = pd.DataFrame(grid_search.cv_results_)
plt.figure(figsize=(8, 6))
sns.lineplot(data=results_df, x='param_classifier__n_estimators', y='mean_test_score', hue='param_classifier__max_depth', palette='viridis')
plt.title('Hyperparameters Tuning Results')
plt.xlabel('Number of Estimators')
plt.ylabel('Mean Test Score')
plt.show()

# The trained model is evaluated on the test set. Confusion matrix is used to visualize the performance of the model. It shows the true positive, true negative, false positive, and false negative predictions of the model.

# i have calculated **model accuracy** on the basis of prediction of test data and also calculated precision,recall and F1 score

In [ ]:
# Predict on the test set using the best model
y_pred = grid_search.predict(X_test)

# Evaluate the model
print("Model Accuracy: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

my model accuracy through random forest classifier model is **95.38 %**.

Now i am calculating features importance after model fitting:

The feature importance gives insight into which features are most useful for making predictions